# BERT Multi Classification 모델 구현 실습 - PyTorch

이 노트북은 뉴스 데이터를 이용한 **BERT 기반 다중 분류 모델**을 PyTorch 코드로 구현합니다.

## 데이터 안내

- 한국언론진흥재단 빅카인즈 올림픽 뉴스 메타데이터 CSV(`한국언론진흥재단_뉴스빅데이터_메타데이터_올림픽_20210808.csv`)를 사용합니다.
- 기사 `본문`을 입력(`news`)으로, `통합 분류1`의 대분류(스포츠·경제·사회·문화·국제·정치·지역·IT_과학 등)를 라벨(`category`)로 변환합니다.
- `category.value_counts()`로 클래스별 데이터 수를 확인하고, Train/Validation/Test 데이터 크기를 확인합니다.
- 뉴스 본문(`news`)과 라벨(`category`)을 사용하는 Dataset 클래스를 정의합니다.
- `kykim/bert-kor-base` Pre-trained 모델을 다운로드하고, 데이터의 고유 클래스 수에 맞춰 `num_labels`를 자동 지정합니다. 이후 Epoch, Batch Size, Weight Decay 등의 하이퍼파라미터를 설정합니다.

> Colab에서 실행할 때는 위 CSV 파일을 런타임에 업로드하면 됩니다. 파일이 없으면 자동으로 업로드 창이 나타납니다.

## 1. 패키지 설치

뉴스 다중 분류도 BERT Tokenizer와 BERT 분류 모델을 사용하므로 `transformers`, `accelerate`, `scikit-learn`을 설치합니다.

In [1]:
# Colab 환경에서 필요한 패키지를 설치합니다.
# transformers: Hugging Face의 BERT 모델과 Tokenizer를 사용하기 위한 라이브러리입니다.
# accelerate: PyTorch 학습 장치 설정을 보조하는 라이브러리로, 최신 transformers와 함께 자주 사용됩니다.
# scikit-learn: 데이터 분리와 평가 지표 계산에 사용합니다.
!pip -q install transformers accelerate scikit-learn

## 2. 라이브러리 불러오기와 재현성 설정

Binary Classification과 동일하게 PyTorch, Pandas, Hugging Face Transformers, Scikit-learn을 사용합니다.

In [2]:
# 운영체제 경로 처리와 파일 확인에 사용하는 표준 라이브러리입니다.
import os

# 난수 시드 고정을 위해 사용하는 표준 라이브러리입니다.
import random

# 배열 연산과 난수 제어를 위해 NumPy를 불러옵니다.
import numpy as np

# CSV 파일을 읽고 표 형태 데이터를 처리하기 위해 Pandas를 불러옵니다.
import pandas as pd

# PyTorch Tensor, 모델, 학습 연산을 사용하기 위해 torch를 불러옵니다.
import torch

# PyTorch Dataset과 DataLoader를 사용하기 위해 필요한 클래스를 불러옵니다.
from torch.utils.data import Dataset, DataLoader

# 데이터셋을 Train/Validation/Test로 나누기 위해 train_test_split을 불러옵니다.
from sklearn.model_selection import train_test_split

# 정확도와 상세 분류 리포트를 계산하기 위해 평가 함수를 불러옵니다.
from sklearn.metrics import accuracy_score, classification_report

# 한국어 BERT Tokenizer를 불러오기 위한 클래스입니다.
from transformers import BertTokenizerFast

# 문장 다중 분류용 BERT 모델을 불러오기 위한 클래스입니다.
from transformers import BertForSequenceClassification

# Transformer 학습에 적합한 AdamW Optimizer를 불러옵니다.
from torch.optim import AdamW

# 학습률 스케줄러를 불러옵니다.
from transformers import get_linear_schedule_with_warmup

# 반복문 진행률을 표시하기 위해 tqdm을 불러옵니다.
from tqdm.auto import tqdm

# 실험 재현성을 위한 난수 시드를 지정합니다.
SEED = 42

# 파이썬 random 모듈의 난수 시드를 고정합니다.
random.seed(SEED)

# NumPy 난수 시드를 고정합니다.
np.random.seed(SEED)

# PyTorch CPU 난수 시드를 고정합니다.
torch.manual_seed(SEED)

# GPU가 있으면 CUDA 난수 시드도 고정합니다.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU가 사용 가능하면 cuda, 아니면 cpu를 학습 장치로 설정합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 학습 장치를 출력합니다.
print("사용 장치:", device)

사용 장치: cuda


## 3. 뉴스 데이터 로드

한국언론진흥재단 빅카인즈 올림픽 뉴스 CSV(`한국언론진흥재단_뉴스빅데이터_메타데이터_올림픽_20210808.csv`)를 Pandas로 읽어 뉴스 본문과 카테고리를 준비합니다.

- **news**: 기사 `본문`
- **category**: `통합 분류1`의 대분류(예: `스포츠>올림픽` → `스포츠`)를 정수 라벨로 변환한 값

이미 `news`/`category` 컬럼을 가진 전처리된 `news.csv`가 있으면 그 파일을 그대로 사용합니다. Colab에서는 파일이 없으면 업로드 창이 나타납니다.

In [ ]:
# Colab에서 직접 파일 업로드를 지원하기 위해 try 문을 사용합니다.
try:
    # Colab 파일 업로드 기능을 불러옵니다.
    from google.colab import files

# Colab이 아닌 환경에서는 google.colab 모듈이 없을 수 있습니다.
except Exception:
    # Colab이 아니면 files 변수를 None으로 둡니다.
    files = None

# 사용 가능한 데이터 파일 후보 경로들입니다.
# 1) 이미 전처리된 news.csv 가 있으면 그대로 사용하고,
# 2) 없으면 한국언론진흥재단 빅카인즈 올림픽 뉴스 원본 CSV를 사용합니다.
CANDIDATE_PATHS = [
    "./data/news.csv",
    "./news.csv",
    "한국언론진흥재단_뉴스빅데이터_메타데이터_올림픽_20210808.csv",
    "./data/한국언론진흥재단_뉴스빅데이터_메타데이터_올림픽_20210808.csv",
]

# 존재하는 첫 번째 후보 경로를 선택합니다.
data_path = None
for candidate in CANDIDATE_PATHS:
    if os.path.exists(candidate):
        data_path = candidate
        break

# 로컬에 파일이 없고 Colab 업로드 기능을 쓸 수 있으면 업로드를 요청합니다.
if data_path is None and files is not None:
    # 사용자에게 뉴스 CSV 파일 업로드 창을 띄웁니다.
    uploaded = files.upload()
    # 업로드된 첫 번째 파일명을 데이터 경로로 사용합니다.
    data_path = next(iter(uploaded.keys()))

# 모든 방법이 실패하면 파일 없음 오류를 발생시킵니다.
if data_path is None:
    raise FileNotFoundError(
        "뉴스 CSV 파일을 찾을 수 없습니다. news.csv 또는 "
        "'한국언론진흥재단_뉴스빅데이터_메타데이터_올림픽_20210808.csv' 파일을 업로드하세요."
    )

# 사용할 데이터 경로를 출력합니다.
print("사용 데이터 파일:", data_path)

# CSV 파일을 Pandas DataFrame으로 읽습니다.
raw = pd.read_csv(data_path)

# 이미 news/category 컬럼이 있는 전처리된 형식이면 그대로 사용합니다.
if "news" in raw.columns and "category" in raw.columns:
    # 필요한 두 컬럼만 추출합니다.
    dataset = raw[["news", "category"]].copy()
    # 라벨 이름은 정수 카테고리를 문자열로 변환해 사용합니다.
    category_names = [str(c) for c in sorted(dataset["category"].dropna().unique().tolist())]

# 빅카인즈 원본 CSV 형식이면 본문과 통합 분류1로 전처리합니다.
else:
    # 뉴스 본문이 들어 있는 컬럼명입니다.
    text_col = "본문"
    # 통합 분류1(대분류>중분류>소분류) 컬럼명입니다.
    label_col = "통합 분류1"

    # 본문과 분류가 모두 존재하는 행만 사용합니다.
    df = raw[[text_col, label_col]].dropna().copy()

    # 통합 분류1에서 '>' 앞의 대분류(예: 스포츠>올림픽 -> 스포츠)만 카테고리로 사용합니다.
    df["category_name"] = df[label_col].astype(str).str.split(">").str[0].str.strip()

    # 카테고리 이름을 정렬하여 정수 인덱스로 매핑합니다.
    category_names = sorted(df["category_name"].unique().tolist())
    name_to_id = {name: idx for idx, name in enumerate(category_names)}

    # 모델 학습에 사용할 news(본문)와 category(정수 라벨) 컬럼을 구성합니다.
    dataset = pd.DataFrame({
        "news": df[text_col].astype(str).values,
        "category": df["category_name"].map(name_to_id).astype(int).values,
    })

# 카테고리 이름과 인덱스 매핑을 출력합니다.
print("카테고리 목록:", dict(enumerate(category_names)))

# 데이터가 정상적으로 준비되었는지 상위 5개 행을 출력합니다.
dataset.head()

## 4. 컬럼 확인, 결측치 제거, 클래스 분포 확인

클래스별 데이터 개수를 확인합니다. 다중 분류에서는 각 클래스의 데이터 수가 균형적인지 확인하는 것이 중요합니다.

In [ ]:
# 데이터셋의 컬럼명을 출력하여 현재 형식을 확인합니다.
print("컬럼 목록:", dataset.columns.tolist())

# 원본 빅카인즈 CSV가 그대로 들어온 경우 news/category 형식으로 변환합니다.
if "category" not in dataset.columns or "news" not in dataset.columns:
    text_col = "본문"
    label_col = "통합 분류1"

    if text_col in dataset.columns and label_col in dataset.columns:
        df = dataset[[text_col, label_col]].dropna().copy()
        df["category_name"] = df[label_col].astype(str).str.split(">").str[0].str.strip()

        category_names = sorted(df["category_name"].unique().tolist())
        name_to_id = {name: idx for idx, name in enumerate(category_names)}

        dataset = pd.DataFrame({
            "news": df[text_col].astype(str).values,
            "category": df["category_name"].map(name_to_id).astype(int).values,
        })

        print("원본 CSV를 news/category 형식으로 변환했습니다.")
        print("카테고리 목록:", dict(enumerate(category_names)))
    else:
        raise ValueError(
            "데이터에는 'news'/'category' 컬럼 또는 "
            "빅카인즈 원본 컬럼 '본문'/'통합 분류1'이 필요합니다."
        )

# news/category 형식이지만 category가 문자 라벨이면 정수 라벨로 변환합니다.
if not pd.api.types.is_numeric_dtype(dataset["category"]):
    category_names = sorted(dataset["category"].dropna().astype(str).unique().tolist())
    name_to_id = {name: idx for idx, name in enumerate(category_names)}
    dataset["category"] = dataset["category"].astype(str).map(name_to_id)
elif "category_names" not in globals():
    category_names = [str(c) for c in sorted(dataset["category"].dropna().unique().tolist())]

print("사용 컬럼 목록:", dataset.columns.tolist())

# 결측치 제거 전 데이터 개수를 저장합니다.
before_count = len(dataset)

# news 또는 category에 결측치가 있는 행을 제거합니다.
dataset = dataset.dropna(subset=["news", "category"]).reset_index(drop=True)

# category를 정수형으로 변환합니다.
dataset["category"] = dataset["category"].astype(int)

# 결측치 제거 후 데이터 개수를 저장합니다.
after_count = len(dataset)

# 제거 전 데이터 수를 출력합니다.
print("결측치 제거 전 데이터 수:", before_count)

# 제거 후 데이터 수를 출력합니다.
print("결측치 제거 후 데이터 수:", after_count)

# 카테고리별 데이터 개수를 출력합니다.
print(dataset["category"].value_counts().sort_index())

# 카테고리가 7인 데이터가 있으면 예시를 출력합니다.
dataset[dataset["category"] == 7].head()

## 5. Train / Validation / Test 데이터 분리

뉴스 카테고리 비율을 유지하기 위해 `stratify=dataset['category']`를 적용합니다.

In [ ]:
# 전체 데이터에서 Train/Test 인덱스를 분리합니다.
train_idx, test_idx, _, _ = train_test_split(
    dataset.index,                 # 분리할 전체 데이터의 인덱스입니다.
    dataset["category"],           # 카테고리 비율을 유지하기 위한 기준 레이블입니다.
    test_size=0.2,                 # 전체 데이터 중 20%를 Test 데이터로 사용합니다.
    stratify=dataset["category"],  # 각 카테고리 비율을 유지합니다.
    random_state=SEED              # 재현성을 위해 난수 시드를 고정합니다.
)

# Train 인덱스에 해당하는 데이터를 선택합니다.
train_set = dataset.iloc[train_idx].reset_index(drop=True)

# Test 인덱스에 해당하는 데이터를 선택합니다.
test_set = dataset.iloc[test_idx].reset_index(drop=True)

# Train 데이터에서 다시 Train/Validation 인덱스를 분리합니다.
train_idx, valid_idx, _, _ = train_test_split(
    train_set.index,                 # 다시 분리할 Train 데이터 인덱스입니다.
    train_set["category"],           # 카테고리 비율을 유지하기 위한 기준 레이블입니다.
    test_size=0.2,                   # Train 데이터 중 20%를 Validation으로 사용합니다.
    stratify=train_set["category"],  # Train/Validation에도 카테고리 비율을 유지합니다.
    random_state=SEED                # 재현성을 위해 난수 시드를 고정합니다.
)

# Validation 데이터를 선택합니다.
valid_set = train_set.iloc[valid_idx].reset_index(drop=True)

# 최종 Train 데이터를 선택합니다.
train_set = train_set.iloc[train_idx].reset_index(drop=True)

# Train 데이터 크기를 출력합니다.
print("Train:", train_set.shape)

# Validation 데이터 크기를 출력합니다.
print("Validation:", valid_set.shape)

# Test 데이터 크기를 출력합니다.
print("Test:", test_set.shape)

## 6. 뉴스 다중 분류 Dataset 클래스 정의

`news`와 `category`를 필드로 사용하는 Dataset 클래스를 정의합니다. 반환 형식은 BERT 입력에 맞게 `input_ids`, `attention_mask`, `labels`입니다.

In [ ]:
# 뉴스 다중 분류용 Dataset 클래스를 정의합니다.
class BertNewsDataset(Dataset):
    # Dataset 객체 생성 시 뉴스 본문, 카테고리, Tokenizer, 최대 길이를 저장합니다.
    def __init__(self, news, category, tokenizer, max_len=128):
        # 뉴스 본문 리스트를 저장합니다.
        self.news = news

        # 카테고리 레이블 리스트를 저장합니다.
        self.category = category

        # BERT Tokenizer를 저장합니다.
        self.tokenizer = tokenizer

        # 모든 문장의 최대 토큰 길이를 저장합니다.
        self.max_len = max_len

    # 전체 데이터 개수를 반환합니다.
    def __len__(self):
        # 뉴스 리스트의 길이를 반환합니다.
        return len(self.news)

    # 특정 index에 해당하는 샘플 하나를 반환합니다.
    def __getitem__(self, index):
        # index 위치의 뉴스 본문을 문자열로 변환합니다.
        text = str(self.news[index])

        # index 위치의 카테고리를 정수로 변환합니다.
        label = int(self.category[index])

        # Tokenizer로 뉴스 본문을 BERT 입력 형식으로 변환합니다.
        encoded = self.tokenizer(
            text,                          # 토큰화할 뉴스 본문입니다.
            add_special_tokens=True,        # [CLS], [SEP] 토큰을 추가합니다.
            max_length=self.max_len,        # 최대 토큰 길이를 지정합니다.
            padding="max_length",          # 짧은 문장을 패딩합니다.
            truncation=True,                # 긴 문장을 최대 길이에 맞게 자릅니다.
            return_attention_mask=True,     # 실제 토큰과 패딩을 구분하는 mask를 반환합니다.
            return_token_type_ids=False,    # 단일 문장 분류이므로 token_type_ids는 사용하지 않습니다.
            return_tensors="pt"             # PyTorch Tensor로 반환합니다.
        )

        # 모델 입력과 정답 레이블을 딕셔너리로 반환합니다.
        return {
            "input_ids": encoded["input_ids"].squeeze(0),           # [1, max_len]을 [max_len]으로 변환합니다.
            "attention_mask": encoded["attention_mask"].squeeze(0), # attention_mask도 [max_len]으로 변환합니다.
            "labels": torch.tensor(label, dtype=torch.long)          # 다중 분류 레이블을 long Tensor로 변환합니다.
        }

## 7. Tokenizer와 Dataset 객체 생성

`kykim/bert-kor-base` Tokenizer를 다운로드한 뒤 Train/Validation/Test Dataset을 생성합니다.

In [ ]:
# 사용할 한국어 BERT 모델 이름을 지정합니다.
bert_model_name = "kykim/bert-kor-base"

# Hugging Face에서 사전 학습된 BERT Tokenizer를 다운로드합니다.
tokenizer = BertTokenizerFast.from_pretrained(bert_model_name)

# BERT 입력 최대 토큰 길이를 지정합니다.
MAX_LEN = 128

# Train DataFrame을 Dataset으로 변환합니다.
train_dataset = BertNewsDataset(
    news=train_set["news"].tolist(),          # Train 뉴스 본문 리스트입니다.
    category=train_set["category"].tolist(),  # Train 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 최대 토큰 길이입니다.
)

# Validation DataFrame을 Dataset으로 변환합니다.
valid_dataset = BertNewsDataset(
    news=valid_set["news"].tolist(),          # Validation 뉴스 본문 리스트입니다.
    category=valid_set["category"].tolist(),  # Validation 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # 같은 BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 같은 최대 토큰 길이입니다.
)

# Test DataFrame을 Dataset으로 변환합니다.
test_dataset = BertNewsDataset(
    news=test_set["news"].tolist(),           # Test 뉴스 본문 리스트입니다.
    category=test_set["category"].tolist(),   # Test 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # 같은 BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 같은 최대 토큰 길이입니다.
)

# 첫 번째 Train 샘플의 구조를 확인합니다.
print(train_dataset[0])

## 8. DataLoader 생성

뉴스 데이터를 미니배치 단위로 모델에 공급하기 위해 DataLoader를 생성합니다.

In [ ]:
# GPU 메모리에 맞게 Batch Size를 지정합니다.
BATCH_SIZE = 16

# Train Dataset을 학습용 DataLoader로 변환합니다.
train_loader = DataLoader(
    train_dataset,          # 학습용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 한 번에 학습할 샘플 수입니다.
    shuffle=True            # 학습 시 데이터 순서를 섞습니다.
)

# Validation Dataset을 검증용 DataLoader로 변환합니다.
valid_loader = DataLoader(
    valid_dataset,          # 검증용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 검증용 Batch Size입니다.
    shuffle=False           # 검증 시 순서를 섞지 않습니다.
)

# Test Dataset을 평가용 DataLoader로 변환합니다.
test_loader = DataLoader(
    test_dataset,           # 평가용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 평가용 Batch Size입니다.
    shuffle=False           # 평가 시 순서를 섞지 않습니다.
)

## 9. Pre-trained BERT 모델 다운로드와 출력 클래스 수 설정

`num_labels`는 분류할 카테고리 개수입니다. 예제에서는 8개 클래스를 사용합니다.

In [ ]:
# 데이터에 존재하는 고유 카테고리 개수를 계산합니다.
NUM_LABELS = dataset["category"].nunique()

# 카테고리 값이 0부터 연속적으로 구성되어 있는지 확인하기 위해 정렬된 목록을 만듭니다.
label_values = sorted(dataset["category"].unique().tolist())

# 현재 데이터의 클래스 목록을 출력합니다.
print("클래스 목록:", label_values)

# 출력층 클래스 수를 출력합니다.
print("NUM_LABELS:", NUM_LABELS)

# 다중 분류용 BERT 모델을 다운로드합니다.
model = BertForSequenceClassification.from_pretrained(
    bert_model_name,      # 사용할 사전 학습 모델 이름입니다.
    num_labels=NUM_LABELS # 다중 분류 클래스 개수입니다.
)

# 모델을 GPU 또는 CPU 장치로 이동합니다.
model = model.to(device)

## 10. Fine-tuning 전략 설정

Binary Classification과 동일하게, BERT의 일부 계층을 고정하여 학습 시간과 메모리 사용량을 줄일 수 있습니다.

In [ ]:
# Fine-tuning 전략을 지정합니다. 0은 전체 학습, 1은 BERT 전체 고정, 2는 pooler만 학습, 3은 마지막 encoder와 pooler만 학습입니다.
tl_strategy = 3

# 전략 1: BERT 본체 전체를 고정합니다.
if tl_strategy == 1:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # 현재 파라미터 이름을 출력합니다.
        print(name)

        # 해당 파라미터를 학습하지 않도록 설정합니다.
        param.requires_grad = False

# 전략 2: pooler를 제외한 BERT 본체를 고정합니다.
elif tl_strategy == 2:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # pooler가 아닌 파라미터만 고정합니다.
        if not name.startswith("pooler"):
            # 해당 파라미터의 gradient 계산을 끕니다.
            param.requires_grad = False

# 전략 3: 마지막 Encoder Layer와 pooler만 학습합니다.
elif tl_strategy == 3:
    # BERT Base 구조의 마지막 encoder layer 이름을 지정합니다.
    last_layer_name = "layer.11"

    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # pooler도 아니고 마지막 layer도 아니면 고정합니다.
        if (not name.startswith("pooler")) and (last_layer_name not in name):
            # 해당 파라미터가 학습되지 않도록 설정합니다.
            param.requires_grad = False

# 학습 가능한 파라미터 수를 계산합니다.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# 전체 파라미터 수를 계산합니다.
total_params = sum(p.numel() for p in model.parameters())

# 학습 가능한 파라미터 비율을 출력합니다.
print(f"학습 가능 파라미터: {trainable_params:,} / 전체 파라미터: {total_params:,}")

## 11. 하이퍼파라미터와 Optimizer 설정

 Epoch, Batch Size, Weight Decay 등의 설정을 PyTorch 학습 루프에 맞게 구성합니다.

In [ ]:
# 전체 학습 Epoch 수를 지정합니다.
EPOCHS = 1

# BERT Fine-tuning에 사용할 학습률을 지정합니다.
LEARNING_RATE = 2e-5

# Weight Decay 정규화 계수를 지정합니다.
WEIGHT_DECAY = 0.01

# Warmup 단계 수를 지정합니다.
WARMUP_STEPS = 0

# 학습 가능한 파라미터만 AdamW Optimizer에 전달합니다.
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), # 고정되지 않은 파라미터만 업데이트합니다.
    lr=LEARNING_RATE,                                      # 학습률입니다.
    weight_decay=WEIGHT_DECAY                              # Weight Decay 계수입니다.
)

# 전체 학습 Step 수를 계산합니다.
total_training_steps = len(train_loader) * EPOCHS

# 선형 학습률 스케줄러를 생성합니다.
scheduler = get_linear_schedule_with_warmup(
    optimizer,                              # 학습률을 조정할 Optimizer입니다.
    num_warmup_steps=WARMUP_STEPS,          # Warmup 단계 수입니다.
    num_training_steps=total_training_steps # 전체 학습 Step 수입니다.
)

## 12. 학습 함수와 평가 함수 정의

다중 분류에서도 BERT 모델은 `labels`를 받으면 내부적으로 CrossEntropyLoss를 계산합니다.

In [ ]:
# 한 Epoch 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    # 모델을 학습 모드로 전환합니다.
    model.train()

    # 전체 loss를 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # 학습 DataLoader에서 배치를 하나씩 가져옵니다.
    for batch in tqdm(data_loader, desc="Training"):
        # input_ids를 학습 장치로 이동합니다.
        input_ids = batch["input_ids"].to(device)

        # attention_mask를 학습 장치로 이동합니다.
        attention_mask = batch["attention_mask"].to(device)

        # labels를 학습 장치로 이동합니다.
        labels = batch["labels"].to(device)

        # 이전 gradient를 초기화합니다.
        optimizer.zero_grad()

        # 모델에 입력을 전달하여 loss와 logits를 계산합니다.
        outputs = model(
            input_ids=input_ids,             # 뉴스 본문 토큰 ID입니다.
            attention_mask=attention_mask,   # 패딩 위치를 구분하는 마스크입니다.
            labels=labels                    # 정답 카테고리입니다.
        )

        # 모델이 계산한 손실값을 가져옵니다.
        loss = outputs.loss

        # 역전파로 gradient를 계산합니다.
        loss.backward()

        # gradient 폭주를 방지하기 위해 gradient norm을 제한합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer가 파라미터를 업데이트합니다.
        optimizer.step()

        # Scheduler가 학습률을 갱신합니다.
        scheduler.step()

        # 배치 loss를 누적합니다.
        total_loss += loss.item()

        # logits에서 가장 큰 값을 가진 클래스 인덱스를 예측값으로 선택합니다.
        preds = torch.argmax(outputs.logits, dim=1)

        # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_labels.extend(labels.detach().cpu().numpy())

        # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss와 정확도를 반환합니다.
    return avg_loss, accuracy

# 모델 평가 함수를 정의합니다.
def evaluate(model, data_loader, device):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 전체 loss를 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # 평가 중에는 gradient 계산을 하지 않습니다.
    with torch.no_grad():
        # 평가 DataLoader에서 배치를 하나씩 가져옵니다.
        for batch in tqdm(data_loader, desc="Evaluating"):
            # input_ids를 평가 장치로 이동합니다.
            input_ids = batch["input_ids"].to(device)

            # attention_mask를 평가 장치로 이동합니다.
            attention_mask = batch["attention_mask"].to(device)

            # labels를 평가 장치로 이동합니다.
            labels = batch["labels"].to(device)

            # 모델에 입력을 전달하여 loss와 logits를 계산합니다.
            outputs = model(
                input_ids=input_ids,           # 뉴스 본문 토큰 ID입니다.
                attention_mask=attention_mask, # 패딩 위치를 구분하는 마스크입니다.
                labels=labels                  # 정답 카테고리입니다.
            )

            # 배치 loss를 누적합니다.
            total_loss += outputs.loss.item()

            # 가장 높은 logit을 가진 클래스를 예측값으로 선택합니다.
            preds = torch.argmax(outputs.logits, dim=1)

            # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_labels.extend(labels.detach().cpu().numpy())

            # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss, 정확도, 전체 정답, 전체 예측을 반환합니다.
    return avg_loss, accuracy, all_labels, all_preds

## 13. 모델 학습

Validation 성능을 확인하면서 뉴스 카테고리 분류 모델을 학습합니다.

In [ ]:
# 지정한 Epoch 수만큼 학습을 반복합니다.
for epoch in range(EPOCHS):
    # 현재 Epoch 번호를 출력합니다.
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Train 데이터로 한 Epoch 학습합니다.
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)

    # Validation 데이터로 모델 성능을 평가합니다.
    valid_loss, valid_acc, _, _ = evaluate(model, valid_loader, device)

    # Train 손실과 정확도를 출력합니다.
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")

    # Validation 손실과 정확도를 출력합니다.
    print(f"Valid Loss: {valid_loss:.4f} | Valid Accuracy: {valid_acc:.4f}")

## 14. Test 데이터 평가

학습이 끝난 후 Test 데이터로 최종 성능을 평가합니다.

In [ ]:
# Test 데이터로 최종 평가를 수행합니다.
test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader, device)

# Test 손실을 출력합니다.
print(f"Test Loss: {test_loss:.4f}")

# Test 정확도를 출력합니다.
print(f"Test Accuracy: {test_acc:.4f}")

# 출력에 사용할 클래스 이름을 카테고리 이름으로 지정합니다.
target_names = [category_names[i] for i in label_values]

# 클래스별 정밀도, 재현율, F1-score를 출력합니다.
print(classification_report(
    test_labels,               # 실제 카테고리 레이블입니다.
    test_preds,                # 모델 예측 카테고리 레이블입니다.
    labels=label_values,       # 평가에 사용할 전체 클래스 목록입니다.
    target_names=target_names, # 출력에 사용할 클래스 이름입니다.
    zero_division=0            # 예측이 없는 클래스의 지표를 0으로 처리합니다.
))

## 15. 새 뉴스 문장 카테고리 예측 함수

학습된 BERT 모델에 새 뉴스 문장을 입력하여 카테고리를 예측합니다.

In [ ]:
# 새 뉴스 문장 하나를 입력받아 카테고리를 예측하는 함수를 정의합니다.
def predict_category(text, model, tokenizer, device, max_len=128):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 입력 뉴스 문장을 BERT 입력 형식으로 변환합니다.
    encoded = tokenizer(
        text,                          # 예측할 뉴스 본문입니다.
        add_special_tokens=True,        # [CLS], [SEP] 토큰을 추가합니다.
        max_length=max_len,             # 최대 토큰 길이를 지정합니다.
        padding="max_length",          # 짧은 문장은 패딩합니다.
        truncation=True,                # 긴 문장은 자릅니다.
        return_attention_mask=True,     # attention_mask를 반환합니다.
        return_tensors="pt"             # PyTorch Tensor로 반환합니다.
    )

    # input_ids를 학습 장치로 이동합니다.
    input_ids = encoded["input_ids"].to(device)

    # attention_mask를 학습 장치로 이동합니다.
    attention_mask = encoded["attention_mask"].to(device)

    # 예측 과정에서는 gradient 계산을 하지 않습니다.
    with torch.no_grad():
        # 모델에 입력을 전달하여 logits를 계산합니다.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # logits를 softmax 확률로 변환합니다.
        probabilities = torch.softmax(outputs.logits, dim=1)

        # 가장 확률이 높은 클래스의 위치를 구합니다.
        predicted_index = torch.argmax(probabilities, dim=1).item()

    # 예측 인덱스를 실제 카테고리 이름으로 변환합니다.
    predicted_label = category_names[predicted_index]

    # 예측 라벨과 클래스별 확률을 반환합니다.
    return predicted_label, probabilities.squeeze(0).detach().cpu().numpy()

# 예측에 사용할 예시 뉴스 문장을 지정합니다.
example_news = "올림픽 대표팀이 결승전에서 승리하며 금메달을 획득했다."

# 예시 뉴스의 카테고리를 예측합니다.
pred_label, probs = predict_category(example_news, model, tokenizer, device, MAX_LEN)

# 입력 문장을 출력합니다.
print("입력 뉴스:", example_news)

# 예측 카테고리를 출력합니다.
print("예측 카테고리:", pred_label)

# 클래스별 확률을 출력합니다.
print("클래스별 확률:", probs)